In [1]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path = self.data.iloc[index, 4]  # 'Preprocessed_Image_paths' is the column containing file paths
        #img_path = img_path.replace("\\",'/')
        image = Image.open(img_path)

        # Convert single-channel image to RGB
        if image.mode == 'L':
            image = image.convert('RGB')

        label = int(self.data.iloc[index, 3])  # 'LABEL' is the column containing labels (0 or 1)

        if self.transform:
            image = self.transform(image)
        
        # Convert PIL image to a NumPy array
        image = np.array(image)

        return image, label

# Load your dataset without normalization for the mean and std calculation
raw_dataset = CustomDataset('Data/sample_preprocessed.csv', transform=None)

# Initialize lists to store channel-wise mean and standard deviation
channel_means = [0.0, 0.0, 0.0]
channel_stds = [0.0, 0.0, 0.0]

# Iterate through your dataset to calculate channel-wise mean and std
for sample in raw_dataset:
    image, _ = sample  # Assuming the label is not used for this calculation
    for i in range(3):  # Assuming your images have 3 channels (R, G, B)
        channel_means[i] += image[i, :, :].mean()
        channel_stds[i] += image[i, :, :].std()

# Calculate the mean and std values across the entire dataset
num_samples = len(raw_dataset)
mean = [c_mean / num_samples for c_mean in channel_means]
std = [c_std / num_samples for c_std in channel_stds]

# Define the transformations
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert to a PyTorch tensor
    transforms.Normalize(mean, std)  # Normalize using the calculated mean and std
])

# Define batch size
batch_size = 64

# Create train and test data loaders
train_dataset = CustomDataset('Data/sample_train.csv', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = CustomDataset('Data/sample_test.csv', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

val_dataset = CustomDataset('Data/sample_validation.csv', transform=transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [2]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Test dataset size: {len(val_dataset)}")

Train dataset size: 2400
Test dataset size: 300
Test dataset size: 300


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the CNN model
class Autism_NadamCNN(nn.Module):
    def __init__(self):
        super(Autism_NadamCNN, self).__init__()
        
        # Convolution Layer 1 (16 filters, Window size = 3x3) + Leaky ReLU
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.leakyrelu1 = nn.LeakyReLU(0.1)
        
        # Maxpooling Layer 1 (Window size=2x2)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout Layer 1 (Proportion = 20%)
        self.dropout1 = nn.Dropout2d(p=0.2)
        
        # Convolution Layer 2 (32 filters, Window size = 3x3) + Leaky ReLU
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.leakyrelu2 = nn.LeakyReLU(0.1)
        
        # Batch Normalization Layer 1
        self.batchnorm1 = nn.BatchNorm2d(32)
        
        # Maxpooling Layer 2 (Window size=2x2)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout Layer 2 (Proportion = 20%)
        self.dropout2 = nn.Dropout2d(p=0.2)
        
        # Convolution Layer 3 (64 filters, Window size = 3x3) + Leaky ReLU
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.leakyrelu3 = nn.LeakyReLU(0.1)
        
        # Batch Normalization Layer 2
        self.batchnorm2 = nn.BatchNorm2d(64)
        
        # Maxpooling Layer 3 (Window size=2x2)
        self.maxpool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout Layer 3 (Proportion = 20%)
        self.dropout3 = nn.Dropout2d(p=0.2)
        
        # Convolution Layer 4 (128 filters, Window size = 3x3) + Leaky ReLU
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.leakyrelu4 = nn.LeakyReLU(0.1)
        
        # Batch Normalization Layer 3
        self.batchnorm3 = nn.BatchNorm2d(128)
        
        # Maxpooling Layer 4 (Window size=2x2)
        self.maxpool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout Layer 4 (Proportion = 20%)
        self.dropout4 = nn.Dropout2d(p=0.2)
        
        # Convolution Layer 5 (256 filters, Window size = 3x3) + Leaky ReLU
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.leakyrelu5 = nn.LeakyReLU(0.1)
        
        # Batch Normalization Layer 4
        self.batchnorm4 = nn.BatchNorm2d(256)
        
        # Maxpooling Layer 5 (Window size=2x2)
        self.maxpool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout Layer 5 (Proportion = 20%)
        self.dropout5 = nn.Dropout2d(p=0.2)
        
        # Flattening Layer
        self.flatten = nn.Flatten()
        
        # Fully Connected Layer (Neurons = 100) + Leaky ReLU
        self.fc1 = nn.Linear(7*7*256, 100)
        
        # Softmax Classifier
        #self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        # Forward pass through the layers
        x = self.leakyrelu1(self.conv1(x))
        x = self.maxpool1(x)
        x = self.dropout1(x)
        
        x = self.leakyrelu2(self.conv2(x))
        x = self.batchnorm1(x)
        x = self.maxpool2(x)
        x = self.dropout2(x)
        
        x = self.leakyrelu3(self.conv3(x))
        x = self.batchnorm2(x)
        x = self.maxpool3(x)
        x = self.dropout3(x)
        
        x = self.leakyrelu4(self.conv4(x))
        x = self.batchnorm3(x)
        x = self.maxpool4(x)
        x = self.dropout4(x)
        
        x = self.leakyrelu5(self.conv5(x))
        x = self.batchnorm4(x)
        x = self.maxpool5(x)
        x = self.dropout5(x)
        
        x = self.flatten(x)
        x = self.leakyrelu5(self.fc1(x))
        #x = self.softmax(x)
        
        return x

# Create an instance of the AutismCNN model
model = Autism_NadamCNN()

In [4]:
import torch.nn as nn

# Define the number of output classes
num_classes = 2

criterion = nn.CrossEntropyLoss()

In [5]:
import torch.optim as optim

# Define the hyperparameters
learning_rate = 0.001  # η
beta1 = 0.9  # β1
beta2 = 0.999  # β2
epsilon = 1e-8  # ε

# Create an instance of the Nadam optimizer with the specified hyperparameters
optimizer = optim.NAdam(model.parameters(), lr=learning_rate, betas=(beta1, beta2), eps=epsilon)


In [6]:
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score

# Load your pretrained model weights here if needed
#model.load_state_dict(torch.load('NAdam_weights.pth'))

# Create empty lists to store training and validation losses and accuracies
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Set the current epoch number to continue from where you left off
current_epoch = 0  # Change this to the epoch you want to continue from

num_epochs = 5  # Set the total number of epochs you want to run

# Define early stopping parameters
patience = 5  # Number of epochs with no improvement after which training will stop
min_delta = 0.001  # Minimum change in validation loss to be considered as improvement
best_val_loss = float('inf')
epochs_without_improvement = 0

# Continue training for additional epochs
for epoch in range(current_epoch, num_epochs):
    model.train()
    running_train_loss = 0.0
    train_preds = []
    train_targets = []

    # Training loop with a progress bar
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs} (Training)'):
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        train_preds.extend(predicted.cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    # Calculate and print the average training loss for this epoch
    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Calculate training accuracy for this epoch
    train_accuracy = accuracy_score(train_targets, train_preds)
    train_accuracies.append(train_accuracy)

    model.eval()  # Set the model to evaluation mode
    running_val_loss = 0.0
    val_preds = []
    val_targets = []

    # Validation loop with a progress bar
    for images, labels in tqdm(val_loader, desc=f'Epoch {epoch + 1}/{num_epochs} (Validation)'):
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_val_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        val_preds.extend(predicted.cpu().numpy())
        val_targets.extend(labels.cpu().numpy())

    # Calculate and print the average validation loss for this epoch
    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Calculate validation accuracy for this epoch
    val_accuracy = accuracy_score(val_targets, val_preds)
    val_accuracies.append(val_accuracy)

    # Early stopping
    if avg_val_loss + min_delta < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), 'SXAI_weights.pth')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping after {patience} epochs without improvement.')
            break

    # Print loss and accuracy for this epoch
    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss: {avg_train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}')
    print(f'Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')

print('Finished Training')


Epoch 1/5 (Validation): 100%|████████████████████████████████████████████████████████████| 5/5 [00:11<00:00,  2.21s/it]


Epoch [1/5], Training Loss: 1.6268, Training Accuracy: 0.57
Validation Loss: 0.7247, Validation Accuracy: 0.63%


Epoch 2/5 (Validation): 100%|████████████████████████████████████████████████████████████| 5/5 [00:07<00:00,  1.49s/it]


Epoch [2/5], Training Loss: 1.0910, Training Accuracy: 0.64
Validation Loss: 2.1366, Validation Accuracy: 0.53%


Epoch 3/5 (Validation): 100%|████████████████████████████████████████████████████████████| 5/5 [00:06<00:00,  1.29s/it]


Epoch [3/5], Training Loss: 0.9559, Training Accuracy: 0.67
Validation Loss: 0.8667, Validation Accuracy: 0.65%


Epoch 4/5 (Validation): 100%|████████████████████████████████████████████████████████████| 5/5 [00:08<00:00,  1.70s/it]


Epoch [4/5], Training Loss: 0.9095, Training Accuracy: 0.69
Validation Loss: 1.0242, Validation Accuracy: 0.69%


Epoch 5/5 (Validation): 100%|████████████████████████████████████████████████████████████| 5/5 [00:06<00:00,  1.24s/it]


Epoch [5/5], Training Loss: 0.8028, Training Accuracy: 0.73
Validation Loss: 0.4667, Validation Accuracy: 0.81%
Finished Training


In [57]:
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, roc_curve, confusion_matrix, classification_report, f1_score, recall_score

# Initialize empty lists to store true labels and predicted class probabilities
true_labels = []
predicted_probs = []

# Set the model to evaluation mode
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        
        # Get the predicted class probabilities using softmax
        predicted_probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())  # Probability of class 1
        
        # Store true labels
        true_labels.extend(labels.cpu().numpy())


# Calculate the accuracy
predicted_labels = [1 if prob >= 0.5 else 0 for prob in predicted_probs]
accuracy = accuracy_score(true_labels, predicted_labels)

# Calculate the AUC
roc_auc = roc_auc_score(true_labels, predicted_probs)

# Calculate precision, recall, and F1 score
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)

# Calculate sensitivity (True Positive Rate)
conf_matrix = confusion_matrix(true_labels, predicted_labels)
sensitivity = conf_matrix[1, 1] / (conf_matrix[1, 1] + conf_matrix[1, 0])

# Calculate specificity (True Negative Rate)
specificity = conf_matrix[0, 0] / (conf_matrix[0, 0] + conf_matrix[0, 1])

# Print the results
print(f'AUC: {roc_auc:.2f}')
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall (Sensitivity): {recall:.2f}')
print(f'Specificity: {specificity:.2f}')
print(f'F1 Score: {f1:.2f}')

AUC: 0.88
Accuracy: 0.80
Precision: 0.84
Recall (Sensitivity): 0.75
Specificity: 0.85
F1 Score: 0.79


In [64]:
import torch
import numpy as np
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# Load your trained model
model = Autism_NadamCNN()
model.load_state_dict(torch.load('SXAI_weights.pth'))  
model.eval()

# Load and preprocess an example MRI image
image_path = 'Data/data_png/32018.nii/32018_148.png'
image = Image.open(image_path).convert('RGB')  # Convert to 3 channels

print("Image size:", image.size)

# Check the type (PIL or NumPy)
if isinstance(image, Image.Image):
    print("Image is in PIL format.")
elif isinstance(image, np.ndarray):
    print("Image is in NumPy format.")
else:
    print("Image format is unknown.")

# Load the original MRI image and convert it to a NumPy array
original_image = np.array(image)
print("Original_image size:", original_image.shape)

# Check the type (PIL or NumPy)
if isinstance(original_image, Image.Image):
    print("Image is in PIL format.")
elif isinstance(original_image, np.ndarray):
    print("Image is in NumPy format.")
else:
    print("Image format is unknown.")

"""
# Display the original image
plt.figure(figsize=(5, 5))
plt.subplot(1, 2, 1)
plt.title('Original Image')
plt.imshow(image)
plt.axis('off')
"""


Image size: (182, 256)
Image is in PIL format.
Original_image size: (256, 182, 3)
Image is in NumPy format.


"\n# Display the original image\nplt.figure(figsize=(5, 5))\nplt.subplot(1, 2, 1)\nplt.title('Original Image')\nplt.imshow(image)\nplt.axis('off')\n"

In [65]:
def preprocess_image(image):
    # Apply transformations
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize the image to match the model's input size
        transforms.ToTensor(),
        transforms.Normalize(mean, std)  # Use the mean and std for normalization
    ])

    # If the input is a NumPy array, convert it to a PIL Image
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    # Preprocess the image and add a batch dimension
    preprocessed_tensor = preprocess(image).unsqueeze(0)

    return preprocessed_tensor

# Preprocess the image and check its shape
preprocessed_image = preprocess_image(image)
print(preprocessed_image.shape)


"""
# Convert the preprocessed image back to a NumPy array for visualization
preprocessed_image_numpy = preprocessed_image.squeeze().permute(1, 2, 0).numpy()

# Display the preprocessed image
plt.subplot(1, 2, 2)
plt.title('Preprocessed Image')
plt.imshow(preprocessed_image_numpy)
plt.axis('off')

# Show both the original and preprocessed images
plt.show()
"""


torch.Size([1, 3, 224, 224])


"\n# Convert the preprocessed image back to a NumPy array for visualization\npreprocessed_image_numpy = preprocessed_image.squeeze().permute(1, 2, 0).numpy()\n\n# Display the preprocessed image\nplt.subplot(1, 2, 2)\nplt.title('Preprocessed Image')\nplt.imshow(preprocessed_image_numpy)\nplt.axis('off')\n\n# Show both the original and preprocessed images\nplt.show()\n"

In [66]:
# Preprocess the image and check its shape
preprocessed_image_1 = preprocess_image(original_image)
print(preprocessed_image_1.shape)

torch.Size([1, 3, 224, 224])


In [60]:
# Define a function to get model predictions for a single image
def predict_with_model(image):
    # Preprocess the image and get the model prediction
    with torch.no_grad():
        input = preprocess_image(image)
        output = model(input)
    return output


In [67]:
def predict_with_model(image):
    # Preprocess the image
    preprocessed_image = preprocess_image(image)

    with torch.no_grad():
        logits = model(preprocessed_image)
        probs = F.softmax(logits, dim=1)
    
    return probs[0].cpu().numpy()


In [68]:
# Initialize the LIME explainer for image classification
explainer = lime_image.LimeImageExplainer()

# Explain the model's prediction for the chosen image
explanation = explainer.explain_instance(
    original_image,
    classifier_fn=predict_with_model,
    top_labels=1,  # Number of top labels to explain (change as needed)
    hide_color=0   # Color to hide (0 means black)

)


  0%|          | 0/1000 [00:00<?, ?it/s]

TypeError: Cannot handle this data type: (1, 1, 182, 3), |u1

In [ ]:
# Visualize the explanation
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0], 
    positive_only=False, 
    hide_rest=False
)

print(temp.shape)
print(mask.shape)

plt.imshow(mark_boundaries(temp / 2 + 0.5, mask))
plt.show()
